In [ ]:
# Imports
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path: sys.path.insert(0, project_root)

from src import *
import scipy.sparse.linalg as spsl
import matplotlib.pyplot as plt
import matplotlib

In [ ]:
# Hamiltonian Parameters
num_qubits = 15
J = 1
h = 1

# Algorithm Parameters
max_energy_level = 3
kd_ratio = 2.5
noise_threshold = 1e-3
epsilon = 1e-4
K_values = list(range(100,550,100))
delta_t = 0.08
num_modmd_observables = 7
num_trials = 20

In [ ]:
# Generate Hamiltonian and get true eigenenergies
hamiltonian = tfim_hamiltonian(num_qubits,J,h)
sparse_hamiltonian = hamiltonian.to_matrix(sparse = True)
hamiltonian_norm = spsl.norm(sparse_hamiltonian,ord = 2)

v, w = spsl.eigsh(sparse_hamiltonian,k=20,which = 'SA')
true_eigenstates = [w[:,i] for i in range(5)]

In [ ]:
# Construct reference state
indices = np.argsort(sparse_hamiltonian.diagonal())
reference_state = bitstring_superposition_state(num_qubits,[bin(indices[i])[2:] for i in range(6)])

# Get evolved reference states
max_K = K_values[-1]
max_d = int(max_K/kd_ratio)
time_evolution_operator = -1j*sparse_hamiltonian*delta_t
evolved_reference_states = spsl.expm_multiply(time_evolution_operator,reference_state,start=0,stop=max_d+max_K+1,num = max_d+max_K+2)

In [ ]:
# ODMD Results
odmd_overlaps = []
odmd_residuals = []
odmd_observables = [SparsePauliOp('I' * num_qubits).to_matrix(sparse=True)]
odmd_evolved_Oi_phi_0_states = [spsl.expm_multiply(1j*sparse_hamiltonian*delta_t, Oi@reference_state,
                                                         start = 0, stop = max_d-1, num = max_d) for Oi in odmd_observables]

X_elements = generate_X_elements(odmd_observables,max_d,max_K,reference_state,evolved_reference_states)

for trial in range(num_trials):

    gaussian_noise = np.random.normal(0,epsilon,size=X_elements.shape) + 1j * np.random.normal(0,epsilon,size=X_elements.shape)
    noisy_X_elements = X_elements + gaussian_noise

    K_overlap_results = []
    K_residual_results = []

    for K in K_values:

        approximate_eigenstates, eigenenergies = modmd_eigenstates(len(odmd_observables),noise_threshold,noisy_X_elements,
                                                    delta_t,K,kd_ratio,max_energy_level,odmd_evolved_Oi_phi_0_states)      
        
        overlaps = []
        residuals = []

        for energy_level in range(len(approximate_eigenstates)):

            overlaps.append(overlap(true_eigenstates[energy_level],approximate_eigenstates[energy_level]))
            residuals.append(residual_norm(sparse_hamiltonian,approximate_eigenstates[energy_level],eigenenergies[energy_level],hamiltonian_norm))
        
        K_overlap_results.append(overlaps)
        K_residual_results.append(residuals)

    odmd_overlaps.append(padded_array(K_overlap_results,max_energy_level+1,-np.inf))
    odmd_residuals.append(padded_array(K_residual_results,max_energy_level+1,-np.inf))

In [ ]:
# MODMD Results
modmd_overlaps = []
modmd_residuals = []

for trial in range(num_trials):

    modmd_observables = random_one_local_paulis(num_qubits,num_modmd_observables)
    modmd_evolved_Oi_phi_0_states = [spsl.expm_multiply(1j*sparse_hamiltonian*delta_t, Oi@reference_state,
                                                         start = 0, stop = max_d-1, num = max_d) for Oi in modmd_observables]

    X_elements = generate_X_elements(modmd_observables,max_d,max_K,reference_state,evolved_reference_states)
    gaussian_noise = np.random.normal(0,epsilon,size=X_elements.shape) + 1j * np.random.normal(0,epsilon,size=X_elements.shape)
    noisy_X_elements = X_elements + gaussian_noise

    K_overlap_results = []
    K_residual_results = []

    for K in K_values:
        approximate_eigenstates, eigenenergies = modmd_eigenstates(len(modmd_observables),noise_threshold,noisy_X_elements,
                                                    delta_t,K,kd_ratio,max_energy_level,modmd_evolved_Oi_phi_0_states)
        overlaps = []
        residuals = []

        for energy_level in range(len(approximate_eigenstates)):
            overlaps.append(overlap(true_eigenstates[energy_level],approximate_eigenstates[energy_level]))
            residuals.append(residual_norm(sparse_hamiltonian,approximate_eigenstates[energy_level],eigenenergies[energy_level],hamiltonian_norm))

        K_overlap_results.append(overlaps)
        K_residual_results.append(residuals)

    modmd_overlaps.append(padded_array(K_overlap_results,max_energy_level+1,-np.inf))
    modmd_residuals.append(padded_array(K_residual_results,max_energy_level+1,-np.inf))

In [ ]:
# Plotting Overlaps
energy_levels = [0,1,2,3]
colors = get_color_set('TFIM')
matplotlib.rcParams.update({'font.size': 14})

odmd_average_overlaps = np.average(odmd_overlaps,0)
modmd_average_overlaps = np.average(modmd_overlaps,0)
for i, energy_level in enumerate(energy_levels):
    plt.semilogy(K_values,1-odmd_average_overlaps[:,i], color = colors[i], linestyle = '--')
    plt.semilogy(K_values,1-modmd_average_overlaps[:,i], color = colors[i], label = f'$n={energy_level}$')

plt.xlabel(r'K ($\propto \text{Simulation Time})$')
plt.ylabel(r'Eigenstate Infidelity')
plt.legend(framealpha = 0)

In [ ]:
# Plotting Residuals
energy_levels = [0,1,2,3]
colors = get_color_set('TFIM')
matplotlib.rcParams.update({'font.size': 14})

odmd_average_residuals = np.average(odmd_residuals,0)
modmd_average_residuals = np.average(modmd_residuals,0)

for i, energy_level in enumerate(energy_levels):
    plt.semilogy(K_values,odmd_average_residuals[:,i], color = colors[i], linestyle = '--')
    plt.semilogy(K_values,modmd_average_residuals[:,i], color = colors[i], label = f'$n={energy_level}$')

plt.xlabel(r'K ($\propto \text{Simulation Time})$')
plt.ylabel('Residual Norm')